In [10]:
import kagglehub
path = kagglehub.dataset_download("fedesoriano/the-boston-houseprice-data")

100%|██████████| 12.3k/12.3k [00:00<00:00, 21.2MB/s]

Extracting files...


In [11]:
import os

# List the contents of the downloaded directory
print(f"Contents of the downloaded directory '{path}':")
for item in os.listdir(path):
    print(item)


Contents of the downloaded directory '/root/.cache/kagglehub/datasets/fedesoriano/the-boston-houseprice-data/versions/1':
boston.csv


Based on the dataset name, the primary data file is likely `boston_house_prices.csv`. I'll load this into a pandas DataFrame.

In [13]:
import pandas as pd
import os

# Construct the full path to the CSV file with the correct filename
data_file_path = os.path.join(path, 'boston.csv')

# Load the dataset into a pandas DataFrame
df = pd.read_csv(data_file_path)

# Display the first 5 rows of the DataFrame
display(df.head())


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(df.drop('MEDV', axis=1), df['MEDV'], test_size=0.2, random_state=42)

In [16]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2_score(y_test, y_pred)

0.6687594935356326

In [17]:
import pickle

# Save the trained model to a file
with open('linear_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved successfully!")

Model saved successfully!


In [18]:
# Create the requirements file for Streamlit
requirements_content = """
streamlit==1.35.0
scikit-learn==1.2.2
numpy==1.25.2
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())

print("requirements.txt created!")

requirements.txt created!


In [20]:
! pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.8 MB/s eta 0:00:00


In [22]:
%%writefile app.py
import streamlit as st
import pickle
import numpy as np

# 1. Set up the page title and description
st.set_page_config(page_title="Boston House Price Predictor", layout="centered")
st.title("🏡 Boston House Price Predictor")
st.write("Enter the neighborhood metrics below to predict the median house value (MEDV).")

# 2. Load the saved Linear Regression model
@st.cache_resource
def load_model():
    with open('linear_model.pkl', 'rb') as f:
        return pickle.load(f)

model = load_model()

# 3. Create input elements for all 13 features
st.subheader("Neighborhood Features")
col1, col2 = st.columns(2)

with col1:
    crim = st.number_input("CRIM (Per capita crime rate)", value=0.006)
    zn = st.number_input("ZN (Proportion of residential land zoned)", value=18.0)
    indus = st.number_input("INDUS (Proportion of non-retail business acres)", value=2.31)
    chas = st.selectbox("CHAS (Charles River dummy variable)", options=[0, 1], index=0)
    nox = st.number_input("NOX (Nitric oxides concentration)", value=0.538)
    rm = st.number_input("RM (Average number of rooms per dwelling)", value=6.57)
    age = st.number_input("AGE (Proportion of owner-occupied units built prior to 1940)", value=65.2)

with col2:
    dis = st.number_input("DIS (Weighted distances to five Boston employment centres)", value=4.09)
    rad = st.number_input("RAD (Index of accessibility to radial highways)", value=1.0)
    tax = st.number_input("TAX (Full-value property-tax rate per $10,000)", value=296.0)
    ptratio = st.number_input("PTRATIO (Pupil-teacher ratio by town)", value=15.3)
    b = st.number_input("B (1000(Bk - 0.63)^2 where Bk is the proportion of blacks by town)", value=396.9)
    lstat = st.number_input("LSTAT (% lower status of the population)", value=4.98)

# 4. Predict button and output logic
if st.button("Predict House Price", type="primary"):
    input_features = np.array([[crim, zn, indus, chas, nox, rm, age, dis, rad, tax, ptratio, b, lstat]])
    prediction = model.predict(input_features)
    predicted_value = round(float(prediction[0]), 2)
    st.success(f"### 💰 Predicted Median House Value: ${predicted_value:,.2f}k")

Writing app.py
